# Interactive Visualization Libraries in Jupyter — SOLUTION

**Complete implementations, alternate approaches, extended practice, and simulation guidance using Plotly.**

This notebook mirrors the Skeleton but provides full working code, professional hover templates, layout polish suitable for data analysis reports, and deeper exploration of when interactivity genuinely helps different audiences.

## Flowchart: Choosing Between Static and Interactive Visualizations

```mermaid
flowchart TD
    Start[Define Audience + Key Questions from Data Analysis Report] --> Needs{What does the audience need?}
    Needs -->|Fixed story, print/PDF, executive summary| Static[Use Static Viz<br/>Matplotlib / Seaborn / Pandas plot<br/>Controlled, reproducible, publication-ready]
    Needs -->|Exploration, filtering, hover details, drill-down, self-service| Interactive[Use Interactive Viz<br/>Plotly (recommended in Jupyter)]
    Interactive --> Library{Which library?}
    Library -->|Fast, expressive, great Jupyter integration| Plotly[Plotly Express first → graph_objects for fine control]
    Library -->|Declarative grammar + rich interactivity| Altair[Altair (if available)]
    Library -->|Highly customizable dashboards| Bokeh[ Bokeh or Panel/Holoviews]
    Plotly --> Enhance[Add hover templates, error info, color encoding, facets, marginals]
    Enhance --> Test[Test with real audience: Can they discover insights themselves?]
    Test --> Deliver[Embed in Jupyter Notebook, HTML report, or lightweight dashboard]
    Static --> Deliver
```


## Setup: Imports and Reusable Datasets

Same data as the skeleton (and previous notebooks) for easy comparison between static and interactive storytelling.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)

regions = ['North America', 'Europe', 'Asia Pacific', 'Latin America']
sales_q1 = np.array([245.0, 312.0, 189.0, 98.0])
sales_q2 = np.array([278.0, 295.0, 210.0, 115.0])
errors_q1 = np.array([18.0, 25.0, 15.0, 9.0])
errors_q2 = np.array([20.0, 22.0, 17.0, 11.0])

df_sales = pd.DataFrame({
    'Region': regions, 'Q1_Sales': sales_q1, 'Q2_Sales': sales_q2,
    'Q1_Error': errors_q1, 'Q2_Error': errors_q2
})
print('=== Regional Sales Data ===')
print(df_sales.to_string(index=False))

novice_scores = np.random.normal(loc=62, scale=14, size=180)
expert_scores = np.random.normal(loc=81, scale=7, size=140)
df_scores = pd.DataFrame({
    'Score': np.concatenate([novice_scores, expert_scores]),
    'Audience': ['Novice']*len(novice_scores) + ['Expert']*len(expert_scores)
})
print('\n=== Score Distributions Ready ===')

content_types = ['Technical Reports', 'Interactive Viz', 'Tutorials', 'Case Studies']
audience_reach = np.array([28, 35, 22, 15])
df_content = pd.DataFrame({'Content Type': content_types, 'Reach %': audience_reach})
print('\n=== Content Reach Data Ready ===')


## 1. Interactive Bar Chart — Complete Solutions

Plotly Express makes beautiful interactive bars in one line. We add professional touches (hover template, layout, text on bars) that work well for both executives and analysts.

In [ ]:
# SOLUTION 1.1: Polished interactive vertical bar chart
fig = px.bar(
    df_sales, x='Region', y='Q1_Sales',
    title='Q1 Sales by Region — Interactive (Hover for exact values)',
    labels={'Q1_Sales': 'Sales (thousands USD)', 'Region': 'Region'},
    color='Region',
    text='Q1_Sales',
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig.update_traces(
    texttemplate='%{text:,.0f}k',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Sales: %{y:,.0f}k USD<br>Error: ±%{customdata[0]:.0f}k<extra></extra>'
)
fig.update_traces(customdata=df_sales[['Q1_Error']].values)

fig.update_layout(
    xaxis_tickangle=-15,
    yaxis_title='Sales (thousands USD)',
    showlegend=False,
    margin=dict(t=60, b=40)
)
fig.show()


## 2. Grouped & Stacked Interactive Bars — Complete

The `barmode` parameter + interactive legend gives users control that static charts cannot match.

In [ ]:
# SOLUTION 2.1: Interactive GROUPED bars (recommended for direct Q1 vs Q2 comparison)
df_long = df_sales.melt(id_vars=['Region'], value_vars=['Q1_Sales', 'Q2_Sales'],
                        var_name='Quarter', value_name='Sales')

fig = px.bar(
    df_long, x='Region', y='Sales', color='Quarter',
    barmode='group',
    title='Q1 vs Q2 Sales by Region — Interactive Grouped Bars<br><sup>Click legend items to toggle quarters</sup>',
    labels={'Sales': 'Sales (thousands USD)'},
    color_discrete_map={'Q1_Sales': '#2E86AB', 'Q2_Sales': '#F18F01'}
)
fig.update_layout(xaxis_tickangle=-15, margin=dict(t=70))
fig.show()


In [ ]:
# SOLUTION 2.2: Interactive STACKED bars (shows total + composition)
fig = px.bar(
    df_long, x='Region', y='Sales', color='Quarter',
    barmode='stack',
    title='Total Sales by Region — Interactive Stacked View',
    labels={'Sales': 'Sales (thousands USD)'}
)
fig.update_layout(xaxis_tickangle=-15)
fig.show()


## 3. Interactive Error Bars — Complete

Plotly Express supports `error_y` natively. The error whiskers are visible, and hover still shows rich detail.

In [ ]:
# SOLUTION 3.1: Interactive bar with visible error bars + rich hover
fig = px.bar(
    df_sales, x='Region', y='Q1_Sales',
    error_y='Q1_Error',
    title='Q1 Sales with Uncertainty — Interactive Error Bars<br><sup>Technical audiences can assess reliability via whiskers + hover</sup>',
    text='Q1_Sales',
    color='Region'
)

fig.update_traces(
    error_y={'visible': True, 'type': 'data', 'color': '#E76F51', 'thickness': 2},
    hovertemplate='<b>%{x}</b><br>Sales: %{y:,.0f}k ± %{error_y:.0f}k USD<extra></extra>'
)
fig.update_layout(xaxis_tickangle=-15, showlegend=False)
fig.show()


## 4. Interactive Histograms with Marginals — Complete

The combination of histogram + marginal box/violin plot is extremely powerful for distribution comparison and is trivial in Plotly.

In [ ]:
# SOLUTION 4.1: Interactive histogram + marginal box (highly recommended for analysts)
fig = px.histogram(
    df_scores, x='Score', color='Audience',
    marginal='box',
    nbins=25,
    title='Interactive Distribution of Comprehension Scores<br><sup>Try box-selecting or lassoing a score range!</sup>',
    opacity=0.65,
    color_discrete_map={'Novice': '#E76F51', 'Expert': '#2A9D8F'}
)
fig.update_layout(bargap=0.08, xaxis_title='Comprehension Score', yaxis_title='Count')
fig.show()


**Alternate with violin marginal** (often clearer for shape comparison):

In [ ]:
# SOLUTION 4.2: Same data with violin marginal (great for shape insight)
fig = px.histogram(
    df_scores, x='Score', color='Audience',
    marginal='violin',
    nbins=25,
    title='Interactive Histogram + Violin Marginal (Shape Comparison)',
    opacity=0.6
)
fig.show()


## 5. Interactive Pie / Donut Charts — Complete

Modern donut style + interactive legend + hover makes pie charts more usable than their static counterparts.

In [ ]:
# SOLUTION 5.1: Interactive donut chart with professional styling
fig = px.pie(
    df_content, names='Content Type', values='Reach %',
    title='Audience Reach by Content Type — Interactive Donut<br><sup>Click legend items to isolate slices</sup>',
    hole=0.35,
    color_discrete_sequence=px.colors.sequential.Tealgrn_r
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Reach: %{value}% of total<extra></extra>'
)
fig.update_layout(legend_title_text='Content Type', margin=dict(t=60))
fig.show()


## 6. Advanced Hover Templates & Report-Ready Layout — Complete

These touches make interactive charts suitable for inclusion in professional data analysis reports or shared Jupyter notebooks.

In [ ]:
# SOLUTION 6.1: Rich hover + executive-friendly layout
fig = px.bar(
    df_sales, x='Region', y='Q1_Sales',
    title='Q1 Regional Performance — Interactive Report View',
    text='Q1_Sales',
    color='Region'
)

fig.update_traces(
    texttemplate='%{text:,.0f}k',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Sales: <b>%{y:,.0f}k USD</b><br>Uncertainty: ±%{customdata[0]:.0f}k<br><i>Hover or click legend to explore</i><extra></extra>'
)
fig.update_traces(customdata=df_sales[['Q1_Error']].values)

fig.update_layout(
    title=dict(text='Q1 Regional Performance — Interactive Report View', x=0.5, font=dict(size=16)),
    xaxis_tickangle=-15,
    yaxis_title='Sales (thousands USD)',
    showlegend=False,
    margin=dict(l=70, r=30, t=70, b=50),
    font=dict(size=11)
)
fig.show()


## Extended Practice & Simulation

Go beyond the basics. These exercises build real fluency with interactive viz for different audiences.

In [ ]:
# EXTENDED SIMULATION + PRACTICE
# 1. Change parameters below, then re-run any solution cell above

# Boost Latin America dramatically (what story does the interactive chart now tell?)
# sales_q1[3] = 220

# 2. Create a faceted version (small multiples) — very powerful in Plotly
# df_facet = df_long.copy()
# fig = px.bar(df_facet, x='Region', y='Sales', color='Quarter', facet_col='Quarter',
#              barmode='group', title='Faceted View — Q1 and Q2 Separately')
# fig.show()

# 3. Normalized interactive histogram (density)
# fig = px.histogram(df_scores, x='Score', color='Audience', histnorm='probability density',
#                    marginal='violin', title='Normalized Interactive Histogram (Density)')
# fig.show()

print('Modify values above and re-run the interactive figure cells to explore impact.')


## Quick Investigation Summary: Plotly vs Other Interactive Libraries

| Library   | Strengths                                      | Weaknesses                              | Best For                              | Jupyter Experience |
|-----------|------------------------------------------------|-----------------------------------------|---------------------------------------|--------------------|
| **Plotly**    | Fast Express API, excellent hover/zoom, error bars, marginals, HTML export | Can feel heavier for very large datasets | Most Jupyter interactive needs, reports, dashboards | Excellent (native) |
| **Altair**    | Beautiful declarative grammar, great for statistical viz | Fewer built-in interactive widgets out of box | Grammar-of-graphics fans, academic/statistical work | Very good (via Vega) |
| **Bokeh**     | Extremely customizable, server apps, streaming | Steeper learning curve for simple charts | Complex custom dashboards, real-time apps | Good               |
| **ipywidgets + matplotlib** | Works with existing static plots, simple widgets | Limited native plot interactivity | Quick interactivity on top of matplotlib/seaborn | Good (if installed) |

**Recommendation from this investigation**: For the vast majority of data analysis work in Jupyter, start with **Plotly Express**. It gives you 80–90% of the interactive value with minimal code, and the resulting figures are perfect for sharing notebooks or converting to HTML reports. Use `graph_objects` when you need pixel-perfect control or custom callbacks.


## Final Reflection: Interactive Viz in Data Analysis Reports

From the audience analysis and report structure documents:
- **Primary collaborator/client**: They often love interactive versions because they can answer their own follow-up questions without asking you for new charts.
- **Executive skimmers**: Still benefit from a strong static version in the Body + an interactive HTML supplement for deeper dives.
- **Technical supervisor**: Interactive charts with rich hover data and visible error information increase confidence in your analysis.

**Best practice**: Create the interactive version first (it forces you to think about the data more deeply), then decide which static snapshots to "freeze" into the formal report Body or Appendix.

You now have a complete toolkit — static (previous notebooks) + interactive (this notebook) — for creating audience-resonant visualizations in Python.

**End of Interactive Visualization Libraries Solution Notebook**